In [42]:
import io
import os


In [43]:
import gzip  
import pickle
import numpy as np
import urllib.request

import matplotlib.pyplot as plt
%matplotlib inline


In [44]:
import boto3
import sagemaker
import sagemaker.amazon.common as smac
from sagemaker.predictor import csv_serializer, json_deserializer
from sagemaker import get_execution_role



In [45]:
DOWNLOADED_FILENAME = 'mnist.pkl.gz'
if not os.path.exists(DOWNLOADED_FILENAME):
    urllib.request.urlretrieve("http://deeplearning.net/data/mnist/mnist.pkl.gz", DOWNLOADED_FILENAME)


In [46]:
with gzip.open('mnist.pkl.gz', 'rb') as f:
    train_set, valid_set, test_set = pickle.load(f, encoding='latin1')

print(len(train_set))

2


In [47]:
vectors = train_set[0].T 
##In order to use PCA, we need to transpose our dataset
# We tipically execute PCA on column data. It reduce aproximation for entire dataset

print(train_set[0].shape)
print(vectors.shape)

(50000, 784)
(784, 50000)


In [48]:
buf = io.BytesIO()

In [49]:
smac.write_numpy_to_dense_tensor(buf, vectors)

In [50]:
buf.seek(0)

0

In [51]:
bucket = 'alexbc-sagemaker-us' #bucket to upload training data
prefix = 'sagemaker/pca'
key = 'recordio-pb-data'
resource = boto3.resource('s3')
bucket_obj=resource.Bucket(bucket)
object_res=bucket_obj.Object(os.path.join(prefix, 'train', key)) #reference to the object which hold the data

train_data_loc = 's3://{}/{}/train/{}'.format(bucket, prefix, key)
print('uploaded training data location{}'.format(train_data_loc))

object_res.upload_fileobj(buf)

uploaded training data locations3://alexbc-sagemaker-us/sagemaker/pca/train/recordio-pb-data


In [52]:
output_location = 's3://{}/{}/output'.format(bucket, prefix)
print('training artifacts will be uploaded to: {}'.format(output_location))


training artifacts will be uploaded to: s3://alexbc-sagemaker-us/sagemaker/pca/output


# Because it's an unsupervised model, we don't specify a label data.

In [53]:
region = boto3.Session().region_name
region

containers = {'us-west-2': '174872318107.dkr.ecr.us-west-2.amazonaws.com/pca:latest',
              'us-east-1': '382416733822.dkr.ecr.us-east-1.amazonaws.com/pca:latest',
              'us-east-2': '404615174143.dkr.ecr.us-east-2.amazonaws.com/pca:latest',
              'eu-west-1': '438346466558.dkr.ecr.eu-west-1.amazonaws.com/pca:latest'}
container = containers[region]

role = get_execution_role()
role

sess = sagemaker.Session()

pca = sagemaker.estimator.Estimator(container,
                                    role, 
                                    train_instance_count=1, 
                                    train_instance_type='ml.c4.xlarge',
                                    output_path=output_location,
                                    sagemaker_session=sess)


In [54]:
pca.set_hyperparameters(feature_dim=50000,
                        num_components=10,
                        subtract_mean=True,
                        algorithm_mode='randomized',
                        mini_batch_size=200)
#randomized algorithm is used for datasets with high dense data (e.g. large number of features para ser usado em algoritmos de aproximação)
# other option is regular algorithm which is used for dataset with sparse data and low to moderate number of features


In [55]:
pca.fit({'train': train_data_loc})

pca_predictor = pca.deploy(initial_instance_count=1,
                           instance_type='ml.m4.xlarge')


INFO:sagemaker:Creating training-job with name: pca-2018-04-28-18-10-24-780


..................................................
Docker entrypoint called with argument(s): train
[04/28/2018 18:14:28 INFO 140333667555136] Reading default configuration from /opt/amazon/lib/python2.7/site-packages/algorithm/resources/default-conf.json: {u'_num_gpus': u'auto', u'_log_level': u'info', u'subtract_mean': u'true', u'force_dense': u'true', u'epochs': 1, u'algorithm_mode': u'regular', u'extra_components': u'-1', u'_kvstore': u'dist_sync', u'_num_kv_servers': u'auto'}
[04/28/2018 18:14:28 INFO 140333667555136] Reading provided configuration from /opt/ml/input/config/hyperparameters.json: {u'algorithm_mode': u'randomized', u'feature_dim': u'50000', u'mini_batch_size': u'200', u'num_components': u'10', u'subtract_mean': u'True'}
[04/28/2018 18:14:28 INFO 140333667555136] Final configuration: {u'num_components': u'10', u'_num_gpus': u'auto', u'_log_level': u'info', u'subtract_mean': u'True', u'force_dense': u'true', u'epochs': 1, u'algorithm_mode': u'randomized', u'feature_di

INFO:sagemaker:Creating model with name: pca-2018-04-28-18-15-02-155


===== Job Complete =====
Billable seconds: 116


INFO:sagemaker:Creating endpoint with name pca-2018-04-28-18-10-24-780


-------------------------------------------------------------------------!

In [56]:

pca_predictor.content_type = 'text/csv'
pca_predictor.serializer = csv_serializer
pca_predictor.deserializer = json_deserializer

train_set[0].shape

train_set[0][:, 0].shape

#extract one row for projection

train_set[0][:, 0]

result = pca_predictor.predict(train_set[0][:, 0])
print(result)



{'projections': [{'projection': [-28.305532455444336, 2.799952507019043, 6.037816524505615, 0.908909797668457, -0.03693847358226776, -0.43731558322906494, -2.042056083679199, 1.9561119079589844, -0.4224138855934143, 3.8184704780578613]}]}


In [57]:
train_set[0].T

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [58]:
#get a projection for all the training set in PCA
eigendigits = []
for array in np.array_split(train_set[0].T, 50):
    result = pca_predictor.predict(array)
    #print(array)
    #eigendigits += [r['projection'] for r in result['projections']]

ClientError: An error occurred (413) when calling the InvokeEndpoint operation: HTTP content length exceeded 5246976 bytes.

In [ ]:
result

In [ ]:
eigendigits = np.array(eigendigits).T

for i in range(10):
    plt.subplot(2, 5, i + 1)
    show_digit(eigendigits[i])



In [59]:
sess.delete_endpoint(pca_predictor.endpoint)


INFO:sagemaker:Deleting endpoint with name: pca-2018-04-28-18-10-24-780
